If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec36_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 36: Building the k-Nearest Neighbors Classifier
v.ekc

Last time: classify by copying your nearest neighbor. Today we build the real thing — let **k neighbors vote** — and then answer the question that makes it science: *how accurate is it?* (Slides: KL Lecture 36 deck.)

**Today's sections:**
1. The Google Science Fair data
2. Distance functions
3. The classifier, function by function
4. Accuracy — the train/test split
5. Standardize if necessary

In [ ]:
from datascience import *
import numpy as np
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

> **Setup cells:** load the CKD data and a jittered copy of the breast-cancer data for plotting.

In [ ]:
ckd = Table.read_table('ckd.csv')
ckd = ckd.relabeled('Blood Glucose Random', 'Glucose').select('Glucose', 'Hemoglobin', 'White Blood Cell Count', 'Class')

In [ ]:
patients = Table.read_table('breast-cancer.csv').drop('ID')

def randomize_column(a):
    return a + np.random.normal(0.0, 0.09, size=len(a))

jittered = Table().with_columns([
        'Bland Chromatin (jittered)', 
        randomize_column(patients.column('Bland Chromatin')),
        'Single Epithelial Cell Size (jittered)', 
        randomize_column(patients.column('Single Epithelial Cell Size')),
        'Class',
        patients.column('Class')
    ])

---
## 1. The Google Science Fair Data 🔬

Brittany Wenger won the 2012 Google Science Fair at 17 with a breast-cancer classifier. Same data, same idea: `Class` 1 = malignant, 0 = benign, plus nine measured attributes. (The jittered scatter just un-stacks overlapping dots.)

In [ ]:
patients = Table.read_table('breast-cancer.csv').drop('ID')
patients.show(5)

In [ ]:
patients.group('Class')

In [ ]:
patients.scatter('Bland Chromatin', 'Single Epithelial Cell Size', group='Class')

In [ ]:
jittered.scatter(0, 1, group='Class')

## Distance ##

As we saw in the slides, the distance between two points $(x_0,y_0)$ and $(x_1,y_1)$ in two dimensions is 

$$D = \sqrt{(x_1-x_0)^2 + (y_1-y_0)^2}$$ 

In three dimensions, the distance between two points $(x_0,y_0,z_0)$ and $(x_1,y_1,z_1)$ is 

$$D = \sqrt{(x_1-x_0)^2 + (y_1-y_0)^2 + (z_1 - z_0)^2}$$


### Activity: Fill in the following functions.

*(These were the in-class fill-ins — shown here complete so you can use this notebook as a reference.)*

In [ ]:
def distance(pt1, pt2):
    """Return the distance between two points, represented as arrays"""
    return np.sqrt(sum((pt2-pt1)**2))

In [ ]:
def row_distance(row1, row2):
    """Return the distance between two numerical rows of a table"""
    row1_array = np.array(row1)
    row2_array = np.array(row2)
    return distance(row1_array,row2_array)

In [ ]:
attributes = patients.drop('Class')
attributes.show(3)

In [ ]:
row_distance(attributes.row(0), attributes.row(1))

In [ ]:
row_distance(attributes.row(0), attributes.row(2))

In [ ]:
row_distance(attributes.row(2), attributes.row(2))

---
## 3. The Classifier, Function by Function

Five small functions, each doing one job — this is how real programs get built.

### 📋 Board Reference — the k-NN pipeline

| Function | Job |
|---|---|
| `distance(pt1, pt2)` | distance between two arrays |
| `row_distance(row1, row2)` | same, for table rows |
| `distances(training, example)` | training table + distance-to-example column |
| `closest(training, example, k)` | the k nearest rows |
| `majority_class(topk)` | the winning vote |
| `classify(training, example, k)` | put it all together |

In [ ]:
def distances(training, example):
    """
    Compute distance between example and every row in training.
    Return training augmented with Distance column
    """
    distances = make_array()
    attributes_only = training.drop('Class')
    
    for row in attributes_only.rows:
        distances = np.append(distances, row_distance(row,example)) # append distance between row and example
        
    return training.with_column('Distance_to_ex', distances)

In [ ]:
example = attributes.row(21)
example

In [ ]:
distances(patients.exclude(21), example).sort('Distance_to_ex')

In [ ]:
def closest(training, example, k):
    """
    Return a table of the k closest neighbors to example
    """
    sorted_distances = distances(training,example).sort('Distance_to_ex')
    return sorted_distances.take(np.arange(k))

In [ ]:
closest(patients.exclude(21), example, 5)

In [ ]:
closest(patients.exclude(21), example, 5).group('Class').sort('count', descending=True)

In [ ]:
def majority_class(topk):
    """
    Return the class with the highest count
    """
    return topk.group('Class').sort('count', descending=True).column(0).item(0)

In [ ]:
def classify(training, example, k):
    """
    Return the majority class among the 
    k nearest neighbors of example
    """
    return majority_class(closest(training, example, k))

In [ ]:
classify(patients.exclude(21), example, 5)

In [ ]:
patients.take(21)

In [ ]:
new_example = attributes.row(10)
classify(patients.exclude(10), new_example, 5)

In [ ]:
patients.take(10)

In [ ]:
another_example = attributes.row(15)
classify(patients.exclude(15), another_example, 5)

In [ ]:
patients.take(15)

## Review of the Steps ##

- `distance(pt1, pt2)`: Returns the distance between the arrays `pt1` and `pt2`
- `row_distance(row1, row2)`: Returns the distance between the rows `row1` and `row2`
- `distances(training, example)`: Returns a table that is `training` with an additional column `'Distance'` that contains the distance between `example` and each row of `training`
- `closest(training, example, k)`: Returns a table of the rows corresponding to the k smallest distances 
- `majority_class(topk)`: Returns the majority class in the `'Class'` column
- `classify(training, example, k)`: Returns the predicted class of `example` based on a `k` nearest neighbors classifier using the historical sample `training`

> **Why `exclude` the example row?** A point is always distance 0 from itself — testing on a row the classifier has already memorized proves nothing. This idea is about to scale up:

---
## 4. Accuracy — the Train/Test Split

Shuffle, split in half: **train** on one half, **test** on the other. The test set is data the classifier has never seen — its accuracy there is an honest estimate.

In [ ]:
patients.num_rows

In [ ]:
shuffled = patients.sample(with_replacement=False) # Randomly permute the rows
training_set = shuffled.take(np.arange(342))
test_set  = shuffled.take(np.arange(342, 683))

In [ ]:
def evaluate_accuracy(training, test, k):
    """Return the proportion of correctly classified examples 
    in the test set"""
    test_attributes = test.drop('Class')
    num_correct = 0
    for i in np.arange(test.num_rows):
        c = classify(training, test_attributes.row(i), k)
        num_correct = num_correct + (c == test.column('Class').item(i))
    return num_correct / test.num_rows

In [ ]:
evaluate_accuracy(training_set, test_set, 5)

In [ ]:
evaluate_accuracy(training_set, test_set, 3)

In [ ]:
evaluate_accuracy(training_set, test_set, 11)

In [ ]:
evaluate_accuracy(training_set, test_set, 1)

### Try It 1: Tune k 😊

1. Evaluate the accuracy with k = 7 and k = 9. Together with the values above (k = 1, 3, 5, 11), is bigger k always better?

In [ ]:
# 1. accuracy at k = 7 and k = 9


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Accuracy hovers in the same high range — bigger k smooths noise but too big drowns out local structure. There's no universally best k; that's why we test.*

```python
# 1.
evaluate_accuracy(training_set, test_set, 7)
evaluate_accuracy(training_set, test_set, 9)
```

</details>

---
## 5. Standardize if Necessary

CKD's attributes live on wildly different scales (glucose in the hundreds, hemoglobin in the teens) — without standardizing, glucose alone decides every distance. Compare the standardized and raw accuracies:

> ⚠️ *Note: the original notebook had a typo (`eshuffled`) that made this comparison silently reuse the wrong table — fixed here.*

In [ ]:
def standard_units(x):
    return (x - np.average(x)) / np.std(x)

In [ ]:
ckd_new = ckd.select('Class').with_columns(
    'Glucose_su', standard_units(ckd.column('Glucose')),
    'Hemoglobin_su', standard_units(ckd.column('Hemoglobin')),
    'WBC_su', standard_units(ckd.column('White Blood Cell Count'))
)

In [ ]:
ckd_new

In [ ]:
shuffled = ckd_new.sample(with_replacement=False) 
training_set = shuffled.take(np.arange(74))
test_set  = shuffled.take(np.arange(74, 148))

In [ ]:
evaluate_accuracy(training_set, test_set, 3)

In [ ]:
shuffled = ckd.sample(with_replacement=False) 
training_set = shuffled.take(np.arange(74))
test_set  = shuffled.take(np.arange(74, 148))

In [ ]:
evaluate_accuracy(training_set, test_set, 3)

---
> **Today's takeaway:** k-NN = distance + vote; honest accuracy needs a test set the classifier never saw; and standardize before measuring distance when scales differ. This is the machinery of the final project's classification questions.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — evaluate a classifier
```python
shuffled = t.sample(with_replacement=False)
training_set = shuffled.take(np.arange(n_train))
test_set = shuffled.take(np.arange(n_train, t.num_rows))
evaluate_accuracy(training_set, test_set, k)
```